# Synthetic tissue: simulate and verify

Builds the synthetic benchmark dataset and checks it. The ground truth (the network
programmed below) is in `simCellsSpace/ground_truth.py`.

**Model.** 4000 cells in four spatial groups: cell type A and B, each either close together
(*Interact*, co-culture) or far apart (*Lone*, mono-culture).

1. A cells express ligand `Gene_50`.
2. B cells near A cells sense it through receptor `Gene_60`, which switches on master
   regulator `Gene_70`.
3. Gene-regulatory network (inside each cell):
   - reactive module: `Gene_70` activates `Gene_71`, `Gene_80` and represses `Gene_72`;
   - autonomous module (independent of signaling): `Gene_73` activates `Gene_74`, `Gene_75`
     and represses `Gene_76`.
4. Feedback: B cells release `Gene_80`, sensed by A cells through `Gene_90`.

Counts are drawn from a negative-binomial model with 40% dropout.

**Output:** `python/sim_cells_datasets_co_mo_simulated_classical.h5ad`
(raw counts in `layers['counts']`, log-normalized `X`, `obs['cell_type']`, `obsm['spatial']`).
Figures: `results/data/`. Seeded (`SEED = 42`), so every run gives the same file.

In [ ]:
# Work from the QuantumXCT/python folder wherever Jupyter was started: the dataset is
# written to python/ and the figures to python/results/data/.
import os, sys
_root = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(_root, "qxct")) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
os.chdir(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
print("working directory:", _root)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy.sparse as sp
from scipy.spatial.distance import cdist
from sklearn.datasets import make_blobs

SEED = 42
OUT = "sim_cells_datasets_co_mo_simulated_classical.h5ad"           # in python/
REFERENCE = os.path.join("simCellsSpace", "reference_histograms.csv")  # stored check values
UPDATE_REFERENCE = False   # True only if you deliberately change the simulation below
FIG_DIR = os.path.join("results", "data")
os.makedirs(FIG_DIR, exist_ok=True)

## 1. Simulator

In [ ]:
def simulate_complex_benchmarking_tissue(
    n_cells=4000,
    n_genes=120,
    lr_forward={50: 60},
    lr_retro={80: 90},
    grn_logic={70: {71: 1.5, 72: -1.5, 80: 2.0}},
    receptor_to_grn={60: 70},
    sparsity_factor=0.4
):
    # --- 1. SETUP SPATIAL SCENARIOS ---
    centroids = [[10, 10], [12, 12], [50, 10], [10, 50]]
    coords, labels = make_blobs(n_samples=n_cells, centers=centroids, cluster_std=2.0, random_state=42)

    role_map = {0: 'Type_A_Interact', 1: 'Type_B_Interact', 2: 'Type_A_Lone', 3: 'Type_B_Lone'}
    cell_types = np.array([role_map[l] for l in labels])

    # --- 2. INITIALIZATION ---
    mu_0 = -2.0
    latent_X = np.random.normal(mu_0, 0.5, (n_cells, n_genes))

    # Adjacency Matrix defines ALL internal relationships
    A_mat = np.zeros((n_genes, n_genes))
    if grn_logic:
        for source, targets in grn_logic.items():
            for target, weight in targets.items():
                A_mat[target, source] = weight

    # --- 3. SPATIAL WEIGHTING ---
    dist_mat = cdist(coords, coords)
    sigma = 4.0
    W = np.exp(-(dist_mat**2) / (2 * sigma**2))
    np.fill_diagonal(W, 0)

    # --- 4. SIMULATION ENGINE ---
    is_a = np.char.startswith(cell_types, 'Type_A')
    is_b = np.char.startswith(cell_types, 'Type_B')

    # STEP A: All Senders (A) produce Forward Ligand (50)
    # This is the "Environmental Input"
    for l_idx in lr_forward.keys():
        latent_X[is_a, l_idx] += 3.5

    # STEP B: Spatial Interaction Pass 1 (A -> B)
    for i in range(n_cells):
        if 'Type_B' in cell_types[i]:
            for l_idx, r_idx in lr_forward.items():
                signal = np.sum(W[i, is_a] * np.exp(latent_X[is_a, l_idx]))
                latent_X[i, r_idx] += np.log1p(signal * 2.0)

                # Receptor 60 triggers Master 70
                if r_idx in receptor_to_grn and latent_X[i, r_idx] > 0.5:
                    ms = receptor_to_grn[r_idx]
                    latent_X[i, ms] += 2.5

    # STEP C: Intracellular Propagation (GRN)
    # This now handles 71, 72, and 80 based on A_mat
    for i in range(n_cells):
        current_snapshot = latent_X[i, :].copy()
        for target_gene in range(n_genes):
            sources = np.where(A_mat[target_gene, :] != 0)[0]
            for source_gene in sources:
                weight = A_mat[target_gene, source_gene]
                if current_snapshot[source_gene] > -1.0:
                    delta = (current_snapshot[source_gene] - mu_0)
                    latent_X[i, target_gene] += weight * delta

    # STEP D: Spatial Interaction Pass 2 (B -> A Feedback)
    for i in range(n_cells):
        if 'Type_A' in cell_types[i]:
            for l_idx, r_idx in lr_retro.items():
                signal = np.sum(W[i, is_b] * np.exp(latent_X[is_b, l_idx]))
                latent_X[i, r_idx] += np.log1p(signal * 2.5)

    # --- 5. STABILIZATION & SAMPLING ---
    latent_X = np.clip(latent_X, -10, 6)
    mu = np.exp(latent_X)
    phi = 0.8
    counts = np.random.negative_binomial(1/phi, (1/phi)/(1/phi + mu))
    dropout_mask = np.random.binomial(1, 1 - sparsity_factor, size=counts.shape)
    counts = counts * dropout_mask

    # --- 6. PACKAGING ---
    adata = ad.AnnData(X=counts.astype(np.float32))
    adata.obs['cell_type'] = cell_types
    adata.obsm['spatial'] = coords
    adata.var_names = [f"Gene_{i}" for i in range(n_genes)]

    return adata

## 2. Simulate and save

In [ ]:
np.random.seed(SEED)   # seeds the expression sampling and dropout (the layout uses random_state=42)
adata = simulate_complex_benchmarking_tissue(
    n_cells=4000,
    n_genes=120,
    lr_forward={50: 60},
    receptor_to_grn={60: 70},
    grn_logic={
        # Module 1: Spatial/Reactive (Triggered by Receptor 60)
        70: {71: 1.5, 72: -1.5, 80: 2.0},
        # Module 2: Autonomous/Constitutive (Happens "despite anything")
        73: {74: 1.5, 75: 2.0, 76: -1.5}
    },
    lr_retro={80: 90},
    sparsity_factor=0.4
)

adata.layers['counts'] = adata.X.copy()          # raw counts: what the analyses read
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.X = sp.csr_matrix(adata.X)
adata.layers['counts'] = sp.csr_matrix(adata.layers['counts'])
adata.write_h5ad(OUT)
print(f"wrote {OUT}: {adata.n_obs} cells x {adata.n_vars} genes (seed {SEED})")
adata

## 3. Verify

Structure checks, then the joint histograms of the QuantumXCT gene panel
(cell type 1 = `Gene_50, Gene_90`; cell type 2 = `Gene_60, Gene_70, Gene_71, Gene_80`)
compared with the stored reference `simCellsSpace/reference_histograms.csv`:

- `seed42` = this seeded build: must match exactly;
- `published` = the file used in the paper (made before seeding, so close but not identical).

In [ ]:
d = ad.read_h5ad(OUT)
expected = {"Type_A_Lone", "Type_B_Lone", "Type_A_Interact", "Type_B_Interact"}
assert set(d.obs["cell_type"].astype(str)) == expected, d.obs["cell_type"].unique()
assert "counts" in d.layers, "raw-count layer 'counts' is missing"
assert d.n_vars == 120 and "spatial" in d.obsm
print(d.obs["cell_type"].value_counts().to_string())
print("structure OK")

PANEL = {"ct1": ["Gene_50", "Gene_90"], "ct2": ["Gene_60", "Gene_70", "Gene_71", "Gene_80"]}
GROUPS = {"ct1_mo": ("Type_A_Lone", "ct1"), "ct2_mo": ("Type_B_Lone", "ct2"),
          "ct1_co": ("Type_A_Interact", "ct1"), "ct2_co": ("Type_B_Interact", "ct2")}

In [ ]:
import matplotlib.pyplot as plt
from qxct.encoding import create_joint_histogram
from qxct.plotting import data_histogram


def fingerprint(adata, group_col, counts):
    """Per group: number of cells, total raw counts, and the joint histogram of the
    QuantumXCT gene panel (gene on = count > 0), i.e. exactly the states the circuits encode."""
    rows = []
    for key, (label, ct) in GROUPS.items():
        cells = (adata.obs[group_col].astype(str) == label).values
        genes = PANEL[ct]
        sub = counts[cells]
        M = sub[:, [adata.var_names.get_loc(g) for g in genes]]
        M = M.toarray() if sp.issparse(M) else np.asarray(M)
        rows.append({"group": key, "item": "n_cells", "value": int(cells.sum())})
        rows.append({"group": key, "item": "total_counts", "value": int(round(float(sub.sum())))})
        hist = dict(create_joint_histogram((M > 0).astype(int))[0])
        for i in range(2 ** len(genes)):
            s = format(i, f"0{len(genes)}b")
            rows.append({"group": key, "item": s, "value": int(hist.get(s, 0))})
    return pd.DataFrame(rows)


def plot_histograms(fp, name):
    """2x2 joint histograms: rows = mono / co-culture, columns = cell type 1 / 2."""
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    for r, cond in enumerate(("mo", "co")):
        for c, ct in enumerate(("ct1", "ct2")):
            key = f"{ct}_{cond}"
            h = fp[(fp.group == key) & ~fp.item.isin(["n_cells", "total_counts"])]
            data_histogram(axes[r, c], dict(zip(h.item, h.value)), len(PANEL[ct]),
                           f"{name} - {GROUPS[key][0]}  ({' '.join(PANEL[ct])})")
    fig.tight_layout()
    path = os.path.join(FIG_DIR, f"{name}_data_hist_grid.png")
    fig.savefig(path, dpi=150)
    plt.show()
    print("saved", path)

In [ ]:
fp = fingerprint(d, "cell_type", d.layers["counts"])
plot_histograms(fp, "synthetic")

In [ ]:
if UPDATE_REFERENCE:
    ref = pd.read_csv(REFERENCE) if os.path.exists(REFERENCE) else fp.rename(columns={"value": "published"})
    ref = ref.drop(columns=["seed42"], errors="ignore").merge(
        fp.rename(columns={"value": "seed42"}), on=["group", "item"], how="right")
    ref[["group", "item", "seed42", "published"]].to_csv(REFERENCE, index=False)
    print("reference updated:", REFERENCE)

ref = pd.read_csv(REFERENCE)
cmp = fp.merge(ref, on=["group", "item"], how="outer")
same = int((cmp["value"] == cmp["seed42"]).sum())
print(f"identical to the seed-{SEED} reference: {same}/{len(cmp)} values")

# how far the seeded build is from the published file, per group (histogram total variation)
h = cmp[~cmp.item.isin(["n_cells", "total_counts"])]
print("distance to the published file (total variation, 0 = identical):")
for g, x in h.groupby("group"):
    print(f"  {g}: {0.5 * np.abs(x.value / x.value.sum() - x.published / x.published.sum()).sum():.4f}")
assert same == len(cmp), cmp[cmp["value"] != cmp["seed42"]]
print("synthetic dataset verified")